# Vectorization & Text Representation

En este notebook se exploran distintas técnicas de vectorización y representación de texto, como One-Hot Encoding, Bag of Words, TF-IDF y CountVectorizer, con el objetivo de analizar cómo influyen en la forma en la que los modelos interpretan la información textual. Estas representaciones permiten transformar los mensajes en formato numérico, necesario para su uso en modelos de Machine Learning. En particular, técnicas como TF-IDF resultan especialmente útiles en modelos clásicos, ya que permiten ponderar la importancia de las palabras en función de su frecuencia, mejorando así la capacidad de discriminación entre clases. Asimismo, se incluye un enfoque de Topic Modeling para explorar de forma no supervisada la estructura temática de los datos

In [1]:
import pickle

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
import random

np.random.seed(42)
random.seed(42)

In [5]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/TRABAJO_NO_ESTRUCTURADOS/data/dataframe_limpio.csv')
df.head(6)

,timestamp,person,message,message_length_words,message_length_chars
0,2022-07-10 20:14:08,Paula,Me estoy agobiando con los grupos,6,33
1,2022-07-10 20:14:12,Paula,Asiq hablamos por aqui,4,22
2,2022-07-10 20:14:12,Carmen,Coño,1,4
3,2022-07-10 20:14:14,Carmen,Gracias,1,7
4,2022-07-10 20:14:15,Carmen,Jajjjajajj,1,10
5,2022-07-10 20:14:38,Carmen,A ver,2,5


Dado que los mensajes de WhatsApp contienen emojis y símbolos que pueden generar ruido en representaciones como Bag of Words o TF-IDF, se realizó una limpieza del texto eliminando estos elementos para los modelos clásicos.

In [6]:
import re

def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'\d+', '', texto)  # quitar números
    texto = re.sub(r'[^\w\s]', '', texto)  # quitar símbolos/emojis
    return texto

df["message"] = df["message"].apply(limpiar_texto)

## One-hot encoding

In [11]:
from sklearn.preprocessing import MultiLabelBinarizer

# Tokenizamos los mensajes
df['tokens'] = df['message'].apply(lambda x: x.split())

# Aplicamos One-Hot Encoding
mlb = MultiLabelBinarizer()
ohe_matrix = mlb.fit_transform(df['tokens'])

# Convertimos a un DataFrame y mostramos las primeras filas
ohe_df = pd.DataFrame(ohe_matrix, columns=mlb.classes_)
print(ohe_df.head())

   _  _fiestas  a  aa  aaa  aaaa  aaaaa  aaaaaa  aaaaaaa  aaaaaaaa  ...  \
0  0         0  0   0    0     0      0       0        0         0  ...   
1  0         0  0   0    0     0      0       0        0         0  ...   
2  0         0  0   0    0     0      0       0        0         0  ...   
3  0         0  0   0    0     0      0       0        0         0  ...   
4  0         0  0   0    0     0      0       0        0         0  ...   

   órganos  última  últimamente  últimas  último  únete  única  únicas  único  \
0        0       0            0        0       0      0      0       0      0   
1        0       0            0        0       0      0      0       0      0   
2        0       0            0        0       0      0      0       0      0   
3        0       0            0        0       0      0      0       0      0   
4        0       0            0        0       0      0      0       0      0   

   útero  
0      0  
1      0  
2      0  
3      0  
4      

One-Hot Encoding convierte cada palabra en un vector binario que tiene tantas dimensiones como el tamaño del vocabulario. Esto significa que cada palabra o símbolo único del corpus se representa con una columna nueva. Si tenemos un gran corpus con muchas palabras o símbolos diferentes, la dimensionalidad de los vectores aumenta considerablemente. Por tanto, no es la mejor opción para nuestro conjunto de datos. Word2Vec es mucho más eficiente, especialmente para capturar relaciones semánticas y representar emojis de manera significativa.

## Bag of Words

In [12]:
from sklearn.feature_extraction.text import CountVectorizer

# crear vectorizador
bow_vectorizer = CountVectorizer(max_features=20, min_df=5)

# aplicar Bag of Words
X_bow = bow_vectorizer.fit_transform(df['message'])

# convertir a dataframe para visualizar
bow_df = pd.DataFrame(X_bow.toarray(), columns=bow_vectorizer.get_feature_names_out())

print("Dimensión matriz BoW:", bow_df.shape)

Dimensión matriz BoW: (67334, 20)


In [13]:
bow_df.head(10)

,con,de,el,en,es,la,las,lo,me,mi,no,pero,por,que,se,si,te,un,ya,yo
0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
9,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0


Se ha aplicado Bag of Words para representar los mensajes del chat de WhatsApp como vectores de frecuencia de palabras. Como vemos en la matriz anterior, las palabras más relevantes artículos o preposiciones como “de”, “la”, “que” o “no”.

Esto indica que el modelo está capturando principalmente palabras funcionales del lenguaje cotidiano, que son muy comunes en conversaciones informales pero aportan poca información sobre el contenido o el estilo de los mensajes.

Por tanto, aunque Bag of Words mejora respecto a One-Hot Encoding al incluir frecuencia, sigue siendo limitado para analizar chats de WhatsApp, ya que no capta el contexto ni el significado del lenguaje.

In [14]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [15]:
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

spanish_stopwords = stopwords.words('spanish')

tfidf_vectorizer = TfidfVectorizer(
    stop_words=spanish_stopwords,
    max_features=3000,
    min_df=5,
    ngram_range=(1,1)
)

X_tfidf_1 = tfidf_vectorizer.fit_transform(df['message'])

le = LabelEncoder()
y = le.fit_transform(df['person'])

In [ ]:
# Almacenamos el tfidf para luego poder utilizarlo para ML
pickle.dump(tfidf_vectorizer, open("/content/drive/MyDrive/TRABAJO_NO_ESTRUCTURADOS/TRABAJO_NO_ESTRUCTURADOS/models/tfidf.pkl","wb"))
pickle.dump(le, open("/content/drive/MyDrive/TRABAJO_NO_ESTRUCTURADOS/TRABAJO_NO_ESTRUCTURADOS/models/le.pkl","wb"))

In [16]:
tfidf_df = pd.DataFrame(
    X_tfidf_1.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

tfidf_df.head(5)

,aa,aaa,aaaa,aaaaa,aaaaaa,aaaaaaa,aaaaaaaa,aaaaaaaaa,aaaaaaay,aaaaaala,...,zara,zity,zona,zonas,zugasti,ánimo,íbamos,ñam,ñe,último
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
feature_names = tfidf_vectorizer.get_feature_names_out()
mean_scores = np.asarray(X_tfidf_1.mean(axis=0)).ravel()

tfidf_importance = pd.DataFrame({
    'palabra': feature_names,
    'score_medio': mean_scores
}).sort_values(by='score_medio', ascending=False)

tfidf_importance.head(20)

,palabra,score_medio
2518,si,0.031456
2932,voy,0.009479
955,esq,0.009076
2277,pues,0.008815
2807,vale,0.008808
358,bueno,0.007817
2182,porq,0.007475
1883,osea,0.007160
297,bien,0.006981
1593,mas,0.006862


In [18]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words=spanish_stopwords,
    max_features=3000,
    min_df=5,
    ngram_range=(1,2)
)

X_tfidf_2 = tfidf_vectorizer.fit_transform(df['message'])

In [19]:
feature_names = tfidf_vectorizer.get_feature_names_out()
mean_scores = np.asarray(X_tfidf_2.mean(axis=0)).ravel()

tfidf_importancia = pd.DataFrame({
    'termino': feature_names,
    'score_medio': mean_scores
})

# Solo bigramas
tfidf_bigramas = tfidf_importancia[
    tfidf_importancia['termino'].str.contains(' ')
].sort_values(by='score_medio', ascending=False)

tfidf_bigramas.head(20)

,termino,score_medio
638,da igual,0.001381
2866,ver si,0.001154
2483,si quereis,0.000943
2487,si quieres,0.000797
2181,pues si,0.000715
224,ay dios,0.000619
481,claudia uni,0.000616
1920,paula echamendi,0.000530
2257,quiero ir,0.000528
2801,vale pues,0.000519


Se ha aplicado TF-IDF para representar los mensajes del chat, ponderando la importancia de cada palabra según su frecuencia en un mensaje y su presencia en el conjunto total del corpus. A diferencia de Bag of Words, este método reduce el peso de las palabras demasiado frecuentes y trata de resaltar términos más representativos.

En nuestro caso, aunque TF-IDF mejora la representación, los resultados siguen mostrando palabras muy propias del lenguaje conversacional de WhatsApp, como “si”, “voy”, “esq”, “pues” o “vale”. Esto indica que, al tratarse de mensajes breves e informales, muchas de las palabras con mayor peso siguen siendo expresiones cotidianas y muletillas del chat, más que términos temáticos.

Al analizar los bigramas, aparecen expresiones algo más útiles, como “da igual”, “si queréis” o “pues sí”, que reflejan mejor la forma natural en la que se comunican las participantes. Aun así, TF-IDF sigue teniendo limitaciones para captar el contexto completo y el significado semántico de los mensajes, por lo que resulta más útil como representación de entrada para modelos de Machine Learning que como herramienta interpretativa principal.

Inicialmente, se ha aplicado TF-IDF considerando cada mensaje como un documento, pero no se ha permitido diferenciar claramente entre las participantes, ya que han predominado palabras muy comunes del lenguaje conversacional.

Por ello, se han agrupado todos los mensajes de cada persona en un único documento con el objetivo de capturar mejor su estilo lingüístico:

In [20]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

# Recreate the TfidfVectorizer that was used for X_tfidf_2
# This is necessary because the original tfidf_vectorizer variable was overwritten.
spanish_stopwords = stopwords.words('spanish')
tfidf_vectorizer_mensaje2 = TfidfVectorizer(
    stop_words=spanish_stopwords,
    max_features=3000,
    min_df=5,
    ngram_range=(1,2)
)

# Fit on the original data to get the correct feature names
tfidf_vectorizer_mensaje2.fit(df['message'])
feature_names = tfidf_vectorizer_mensaje2.get_feature_names_out()

people = df['person'].unique()

for person in people:

    # índices de los mensajes de esa persona
    indices = df.index[df['person'] == person]

    # calcular media tfidf
    # Ensure X_tfidf_2 is used, not X_tfidf_1 which has a different dimension
    scores = np.asarray(X_tfidf_2[indices].mean(axis=0)).ravel()

    # ordenar
    top_idx = scores.argsort()[-15:][::-1]

    palabras = [feature_names[i] for i in top_idx]

    print("\nPalabras características de", person)
    print(palabras)


Palabras características de Paula
['si', 'voy', 'bien', 'ver', 'vale', 'pues', 'mas', 'osea', 'esq', 'carmen', 'bueno', 'enplan', 'nose', 'porq', 'mañana']

Palabras características de Carmen
['si', 'osea', 'esq', 'porq', 'pues', 'bueno', 'mas', 'igual', 'vale', 'jaja', 'voy', 'ay', 'vd', 'asi', 'sii']

Palabras características de Angela
['si', 'tb', 'pues', 'esq', 'voy', 'tia', 'vale', 'bien', 'sii', 'bueno', 'xq', 'ver', 'oye', 'así', 'creo']

Palabras características de Claudia
['si', 'voy', 'porq', 'vale', 'bueno', 'carmen', 'plan', 'tia', 'asi', 'pues', 'esq', 'mas', 'lit', 'da', 'luego']


In [21]:
docs_person = df.groupby("person")["message"].apply(lambda x: " ".join(x))

docs_person

,message
person,
Angela,llamo para reservar lo acabo d ver xq no sale ...
Carmen,coño gracias jajjjajajj a ver si solo hay un v...
Claudia,o dios vamos al cherru peecas y ya esta esta s...
Paula,me estoy agobiando con los grupos asiq hablamo...


In [22]:
docs = docs_person.values
people = docs_person.index

from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

spanish_stopwords = stopwords.words("spanish")

tfidf_vectorizer = TfidfVectorizer(
    stop_words=spanish_stopwords,
    max_features=2000,
    ngram_range=(1,2)
)

X_tfidf = tfidf_vectorizer.fit_transform(docs)

In [23]:
print(people)

Index(['Angela', 'Carmen', 'Claudia', 'Paula'], dtype='object', name='person')


In [24]:
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
tfidf_df.head()

,aa,aa vale,aaa,aaa vale,aaaa,aaaaa,aaaaaa,aaaaaaa,aaaaaaaa,aaaaay,...,yess,yesss,yessss,you,zambrano,zara,zity,zona,ánimo,ñam
0,0.000000,0.000000,0.001991,0.000664,0.007963,0.008626,0.006636,0.007299,0.005972,0.000000,...,0.001628,0.004340,0.003255,0.000664,0.000543,0.000543,0.001628,0.007595,0.011475,0.003318
1,0.078324,0.007427,0.043731,0.003280,0.031158,0.015306,0.009293,0.002733,0.002733,0.013703,...,0.008938,0.015195,0.010279,0.000000,0.003128,0.003128,0.001788,0.004916,0.000000,0.000000
2,0.010934,0.003364,0.011575,0.003404,0.012256,0.010894,0.007490,0.004085,0.000681,0.000000,...,0.005567,0.007237,0.002783,0.006128,0.000557,0.003897,0.001670,0.006123,0.001682,0.011575
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000484,0.003385,0.002418,0.003549,0.004836,0.002418,0.001934,0.004352,0.000000,0.009464


In [25]:
import numpy as np
import pandas as pd

feature_names = tfidf_vectorizer.get_feature_names_out()

for i, person in enumerate(people):

    scores = X_tfidf[i].toarray().flatten()
    top_idx = scores.argsort()[-20:][::-1]

    words = [feature_names[j] for j in top_idx]

    print("\nPalabras características de", person)
    print(words)


Palabras características de Angela
['si', 'xq', 'esq', 'pues', 'voy', 'tb', 'cn', 'tia', 'vale', 'ver', 'bien', 'así', 'igual', 'bueno', 'luego', 'creo', 'va', 'ir', 'ahora', 'osea']

Palabras características de Carmen
['si', 'porq', 'esq', 'osea', 'pues', 'mas', 'bueno', 'igual', 'vale', 'voy', 'ver', 'jaja', 'asi', 'vd', 'luego', 'tb', 'ahora', 'sino', 'ay', 'aa']

Palabras características de Claudia
['si', 'porq', 'voy', 'esq', 'vale', 'mas', 'da', 'pues', 'tia', 'luego', 'asi', 'bueno', 'plan', 'carmen', 'va', 'bien', 'ir', 'tb', 'nose', 'quiero']

Palabras características de Paula
['si', 'voy', 'porq', 'ver', 'mas', 'pues', 'enplan', 'bien', 'esq', 'osea', 'vale', 'ir', 'va', 'bueno', 'carmen', 'mañana', 'asiq', 'vamos', 'asi', 'nose']


Podemos observar que siguen apareciendo expresiones habituales como “si”, “vale” o “pues”, lo que indica que TF-IDF no ha mejorado la capacidad de diferenciación en este contexto.

## Word2vec

In [8]:
import sys
!{sys.executable} -m pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.3 MB/s eta 0:00:00


In [9]:
import gensim
from gensim.models import Word2Vec
import pandas as pd

# Tokenizar los mensajes (dividir en palabras)
df['tokens'] = df['message'].apply(lambda x: x.split())

# Ver las primeras filas con las palabras tokenizadas
print(df[['message', 'tokens']].head())

                             message                                    tokens
0  me estoy agobiando con los grupos  [me, estoy, agobiando, con, los, grupos]
1             asiq hablamos por aqui               [asiq, hablamos, por, aqui]
2                               coño                                    [coño]
3                            gracias                                 [gracias]
4                         jajjjajajj                              [jajjjajajj]


In [27]:
# Entrenar el modelo Word2Vec
model = Word2Vec(sentences=df['tokens'], vector_size=100, window=5, min_count=1, workers=4)

100 dimensiones es un buen punto de partida y window=5 es un tamaño de ventana es adecuado para capturar relaciones semánticas sin ser demasiado específico.
Min_count=1: Aunque idealmente se querría un valor más alto (para eliminar palabras raras), usar min_count=1 es útil para ver cómo se comporta el modelo en este contexto y experimentar con la limpieza de datos más adelante.

In [28]:
# Guardar el modelo entrenado
model.save("word2vec_model")

In [ ]:
# Ver las palabras más similares a "cenar"
similar_words = model.wv.most_similar('cenar', topn=5)
print(f"Palabras más similares a 'cenar': {similar_words}")

Palabras más similares a 'cenar': [('comer', 0.9874002933502197), ('volver', 0.9857013821601868), ('quedar', 0.9853779077529907), ('llegar', 0.9845747947692871), ('salir', 0.9827242493629456)]


In [ ]:
# Ver las palabras más similares a "cenar"
similar_words = model.wv.most_similar('apetece', topn=5)
print(f"Palabras más similares a 'aburrido': {similar_words}")

Palabras más similares a 'aburrido': [('ape', 0.9916349053382874), ('decis', 0.9903631210327148), ('gusta', 0.9850640892982483), ('renta', 0.9813259243965149), ('pongo', 0.9808101654052734)]


Mientras que TF-IDF da más peso a las palabras que aparecen con frecuencia, Word2Vec captura relaciones semánticas al identificar que, por ejemplo, "cenar", "comer", "salir" y "quedar" están relacionados no solo por su frecuencia, sino por su contexto compartido. Este tipo de relaciones es fundamental para comprender cómo las personas en un chat de WhatsApp usan el lenguaje

In [ ]:
# Función para obtener el vector promedio de un mensaje
def get_average_word2vec(tokens, model):
    # Filtrar las palabras que están en el vocabulario de Word2Vec
    valid_words = [word for word in tokens if word in model.wv.key_to_index]
    if len(valid_words) == 0:
        return np.zeros(model.vector_size)  # Si no hay palabras en el vocabulario, devuelve un vector cero
    # Promediar los vectores de las palabras válidas
    return np.mean([model.wv[word] for word in valid_words], axis=0)

# Aplicar la función a cada mensaje
df['word2vec_vector'] = df['tokens'].apply(lambda x: get_average_word2vec(x, model))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# X = los vectores de los mensajes (vectores promediados de Word2Vec)
X = np.array(df['word2vec_vector'].tolist())  # Convertir las listas en un array numpy

# Y = las etiquetas (quién escribió el mensaje)
y = df['person']

# Dividir en datos de entrenamiento y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear el modelo de regresión logística
model = LogisticRegression(max_iter=1000)

# Entrenar el modelo
model.fit(X_train, y_train)

# Predecir en el conjunto de test
y_pred = model.predict(X_test)

# Evaluar el modelo
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      Angela       0.42      0.12      0.18      2310
      Carmen       0.36      0.25      0.29      3481
     Claudia       0.33      0.60      0.43      4140
       Paula       0.39      0.34      0.36      3536

    accuracy                           0.36     13467
   macro avg       0.38      0.32      0.32     13467
weighted avg       0.37      0.36      0.33     13467



## Topic Modeling con LDA: Identificación de temas en la conversación

Se ha aplicado un modelo de Topic Modeling mediante LDA con el objetivo de identificar los principales temas presentes en la conversación, utilizando una representación basada en CountVectorizer y eliminando palabras vacías en español.

In [32]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords

spanish_stopwords = stopwords.words('spanish')

# Usamos CountVectorizer para Topic Modeling
count_vectorizer = CountVectorizer(
    stop_words=spanish_stopwords,
    max_features=3000,
    min_df=5,
    ngram_range=(1,2),
    token_pattern=r'(?u)\b[a-zA-ZáéíóúñÁÉÍÓÚÑ]+\b'
)

X_count = count_vectorizer.fit_transform(df['message'])

In [33]:
from sklearn.decomposition import LatentDirichletAllocation

# Número de temas
n_topics = 5

# Aplicamos LDA
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, learning_method='batch')

lda.fit(X_count)

LatentDirichletAllocation(n_components=5, random_state=42)

In [34]:
def print_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_features_idx = topic.argsort()[-n_top_words:][::-1]
        top_features = [feature_names[i] for i in top_features_idx]
        print(f"\nTema {topic_idx + 1}:")
        print(", ".join(top_features))

print_topics(lda, count_vectorizer.get_feature_names_out(), n_top_words=20)


Tema 1:
pues, si, vamos, quiero, mejor, tb, dia, va, angy, siii, ir, vd, menos, hoy, mal, hacer, vale, entonces, podemos, piso

Tema 2:
q, bien, esq, ay, carmen, asi, plan, oye, va, dios, lit, vas, genial, tarde, parece, habia, clau, todas, noche, dice

Tema 3:
q, si, voy, ver, creo, tia, dicho, cosas, q si, creo q, osea, tal, hacer, bro, esq, q q, si q, tiempo, hora, claro

Tema 4:
m, q, mas, bueno, igual, vale, luego, da, mañana, super, osea, ayer, q m, sisi, siempre, hecho, porq, siiii, nose, perfe

Tema 5:
porq, casa, d, hace, sino, puedo, sii, ahora, dos, esq, solo, justo, pasa, digo, cosa, luego, asiq, aun, paula, dias


In [ ]:
# Obtener la probabilidad de cada mensaje para cada tema
topic_probs = lda.transform(X_count)

# Asignar el tema más probable a cada mensaje
df['topic'] = topic_probs.argmax(axis=1)

# Ver los primeros mensajes con su tema asignado
print(df[['person', 'message', 'topic']].head(10))

   person                              message  topic
0   Paula    me estoy agobiando con los grupos      5
1   Paula               asiq hablamos por aqui      1
2  Carmen                                 coño      2
3  Carmen                              gracias      6
4  Carmen                           jajjjajajj      0
5  Carmen                                a ver      7
6  Carmen    si solo hay un vicio y es viernes      6
7  Carmen                 estara petado yo cre      2
8   Paula        yo voy al baño y voy para all      9
9  Carmen  aunq la gente suele pedir por glovo      2


Los resultados obtenidos no son fácilmente interpretables, ya que los temas generados no presentan una coherencia clara. Esto se debe principalmente a la naturaleza del dataset, compuesto por mensajes cortos, informales y con poco contexto, lo que dificulta la identificación de temas bien definidos.

En apartados posteriores se analizan características más específicas de los mensajes y de los participantes, donde se obtiene una mejor representación del comportamiento lingüístico.